# Face model v2 — round 6: counter the FER+ neutral bias
Round 5's FER+ model has the best FER+ test score (0.708 vs v1 0.569) but answers *neutral* on half of the acted RAVDESS clips (expression recall 0.55 vs v1 0.87).
This round lowers the neutral share of the targets (`--neutral-down`), the neutral loss weight (`--neutral-w`), and tries choosing the checkpoint with
RAVDESS actors 1-2 as a second check (`--select combo`). Actors 3-4 stay unseen for the final video test. Every run logs RAVDESS-dev scores per epoch.

Needs in `MyDrive/face-v2/`: `Face-Lab-code-r6.zip`, the FER2013 zip, `out/face_v2_results_r5.zip`. Runtime → T4 GPU.

In [ ]:
import shutil, subprocess
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout)
else:
    raise SystemExit('NO GPU: Runtime > Change runtime type > T4 GPU, then Run all again')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, os, zipfile
D='/content/drive/MyDrive/face-v2'
os.makedirs('/content/Main_demo', exist_ok=True)
zipfile.ZipFile(f'{D}/Face-Lab-code-r6.zip').extractall('/content/Main_demo')
fer = [p for p in glob.glob(f'{D}/*.zip') if 'Face-Lab' not in p][0]
zipfile.ZipFile(fer).extractall('/content/Main_demo/FER2013-Data')
zipfile.ZipFile(f'{D}/out/face_v2_results_r5.zip').extractall('/content/Main_demo/Face-Lab')   # rounds 1-5 results/ for the table
for s in ('train','test'):
    print(s, sum(len(f) for _,_,f in os.walk(f'/content/Main_demo/FER2013-Data/{s}')))   # expect 28709 / 7178
%cd /content/Main_demo/Face-Lab
print(sorted(os.listdir('results')))

In [ ]:
BASE = '--arch efficientnet_b0 --epochs 20 --head-epochs 1 --unfreeze 0.0 --split-seed 42 --seed 42'
FP = '--ferplus 1 --smooth 0'
EXPS = {
  'v2_r6_nd05':        f'--res 96 {FP} --neutral-down 0.5 --select ferplus',
  'v2_r6_nd03':        f'--res 96 {FP} --neutral-down 0.3 --select ferplus',
  'v2_r6_nw05':        f'--res 96 {FP} --neutral-w 0.5 --select ferplus',
  'v2_r6_nd05_combo':  f'--res 96 {FP} --neutral-down 0.5 --select combo --sel-w 0.5',
  'v2_r6_fp1_combo':   f'--res 96 {FP} --select combo --sel-w 0.5',
  'v2_r6_nd05_res112': f'--res 112 {FP} --neutral-down 0.5 --select ferplus',
}
for tag, extra in EXPS.items():
    cmd = f'python train.py {BASE} {extra} --tag {tag}'
    print('>>', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    lines = r.stdout.strip().splitlines()
    print(chr(10).join(lines[-3:]) if lines else '', r.stderr[-800:] if r.returncode else '', flush=True)


In [ ]:
import json, glob
print('reference (round 5, v2_fps_res96_s42): FER+ val F1 0.707; RAVDESS all actors expr recall 0.55 (v1 0.87)')
print('run'.ljust(20), 'fp_F1   fp_acc  fear   sad    angry  | RAV-dev exprRec neuKept F1     ep  min')
for f in sorted(glob.glob('results/v2_r6_*.json')):
    r = json.load(open(f)); fp = r['val_ferplus']; pc = fp['per_class']; rv = r['rav_dev']
    print(r['tag'].ljust(20), '%.4f  %.4f  %.3f  %.3f  %.3f  |  %.3f   %.3f   %.3f  %3d  %4.1f' % (fp['macro_f1'], fp['accuracy'], pc['fear']['f1'], pc['sad']['f1'], pc['angry']['f1'],
          rv['rav_dev_expr_recall'], rv['rav_dev_neutral_kept'], rv['rav_dev_macro_f1'], r['best_epoch'], r['train_minutes']))

In [ ]:
OUT='/content/drive/MyDrive/face-v2/out'
shutil.make_archive('/content/face_v2_results_r6','zip','/content/Main_demo/Face-Lab','results')
shutil.copy('/content/face_v2_results_r6.zip', OUT)
for t in EXPS:
    if os.path.exists(f'runs/{t}/best.pt'):
        os.makedirs(f'{OUT}/{t}', exist_ok=True)
        b = open(f'runs/{t}/best.pt','rb').read()
        for i in range(0, len(b), 9000000):                     # parts < 10 MB so they can be pulled back to the laptop
            open(f'{OUT}/{t}/best.pt.part{i // 9000000}','wb').write(b[i:i + 9000000])
print(sorted(os.listdir(OUT)))